In [1]:
import os
import pandas as pd
import re

In [2]:
city = 'HAN'
df = pd.read_csv(f'{city}_merged.csv')
df.head(3)

,brand,price,start_time,start_day,end_time,end_day,trip_time,take_place,destination,hand_luggage,checked_baggage,crawl_date
0,Bamboo Airways,2.852.590 VND/khách,21:05,02 thg 5,23:15,02 thg 5,2h 10m,TP HCM (SGN)\nSân bay Tân Sơn Nhất,Hà Nội (HAN)\nSân bay Nội Bài,Hành lý xách tay 7 kg,Hành lý 0 kg,01-05-2025
1,VietJet Air,2.770.744 VND/khách,20:05,02 thg 5,22:10,02 thg 5,2h 5m,TP HCM (SGN)\nSân bay Tân Sơn Nhất\nNhà ga 1,Hà Nội (HAN)\nSân bay Nội Bài\nNhà ga 1,Hành lý xách tay 7 kg,Hành lý 0 kg,01-05-2025
2,VietJet Air,2.770.744 VND/khách,18:45,02 thg 5,20:55,02 thg 5,2h 10m,TP HCM (SGN)\nSân bay Tân Sơn Nhất\nNhà ga 1,Hà Nội (HAN)\nSân bay Nội Bài\nNhà ga 1,Hành lý xách tay 7 kg,Hành lý 0 kg,01-05-2025


# PREPROCESSING

## Mark id

In [3]:
group_cols = ['brand', 'start_time', 'start_day', 'end_time', 'end_day', 'trip_time']

df['id'] = None

for i, (group_values, group_df) in enumerate(df.groupby(group_cols), start=1):
    group_label = f'{city}{i:04d}' 
    df.loc[group_df.index, 'id'] = group_label

In [4]:
df['id'].nunique()

1353

## duplicate

In [5]:
duplicates = df[df.duplicated()]
len(duplicates)

243

In [6]:
df = df.drop_duplicates(keep='first').reset_index(drop=True)
df.shape, df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19359 entries, 0 to 19358
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   brand            19359 non-null  object
 1   price            19359 non-null  object
 2   start_time       19359 non-null  object
 3   start_day        19359 non-null  object
 4   end_time         19359 non-null  object
 5   end_day          19359 non-null  object
 6   trip_time        19359 non-null  object
 7   take_place       19359 non-null  object
 8   destination      19359 non-null  object
 9   hand_luggage     14599 non-null  object
 10  checked_baggage  14599 non-null  object
 11  crawl_date       19359 non-null  object
 12  id               19359 non-null  object
dtypes: object(13)
memory usage: 1.9+ MB


((19359, 13), None)

## brand

In [7]:
df['brand'].nunique(), df['brand'].unique()

(9,
 array(['Bamboo Airways', 'VietJet Air', 'Vietnam Airlines',
        'Vietravel Airlines', 'VietJet Air, Bamboo Airways',
        'Vietnam Airlines, VietJet Air',
        'Vietnam Airlines, Bamboo Airways',
        'VietJet Air, Vietnam Airlines', 'Bamboo Airways, VietJet Air'],
       dtype=object))

In [8]:
len(df)

19359

drop 1-stop flights

In [9]:
direct_flights = ['Bamboo Airways', 'VietJet Air', 'Vietnam Airlines', 'Vietravel Airlines']
df = df[df['brand'].isin(direct_flights)]
df['brand'].nunique(), df['brand'].unique()

(4,
 array(['Bamboo Airways', 'VietJet Air', 'Vietnam Airlines',
        'Vietravel Airlines'], dtype=object))

In [10]:
len(df)

19341

In [11]:
df['crawl_date'].nunique(), df['crawl_date'].unique()

(33,
 array(['01-05-2025', '02-05-2025', '03-05-2025', '04-05-2025',
        '05-05-2025', '06-05-2025', '07-04-2025', '07-05-2025',
        '08-04-2025', '08-05-2025', '09-04-2025', '09-05-2025',
        '10-04-2025', '11-04-2025', '12-04-2025', '13-04-2025',
        '14-04-2025', '15-04-2025', '16-04-2025', '17-04-2025',
        '18-04-2025', '19-04-2025', '20-04-2025', '21-04-2025',
        '22-04-2025', '23-04-2025', '24-04-2025', '25-04-2025',
        '26-04-2025', '27-04-2025', '28-04-2025', '29-04-2025',
        '30-04-2025'], dtype=object))

## price

In [12]:
df['price'].head()

0    2.852.590 VND/khách
1    2.770.744 VND/khách
2    2.770.744 VND/khách
3    2.770.744 VND/khách
4    2.770.744 VND/khách
Name: price, dtype: object

In [13]:
# Clean and convert the price column
df['price'] = df['price'].str.extract(r'([\d\.]+)')
df['price'] = df['price'].str.replace('.', '', regex=False).astype(int)
df['price'].head()

0    2852590
1    2770744
2    2770744
3    2770744
4    2770744
Name: price, dtype: int64

## time

In [14]:
df['start_time'].nunique(), df['start_time'].dtype, df['end_time'].nunique(), df['end_time'].dtype

(134, dtype('O'), 138, dtype('O'))

In [15]:
df['start_hour'] = df['start_time'].str.split(':').str[0].astype(int)
df['end_hour'] = df['end_time'].str.split(':').str[0].astype(int)
df[['start_time', 'start_hour', 'end_time', 'end_hour']].head()

,start_time,start_hour,end_time,end_hour
0,21:05,21,23:15,23
1,20:05,20,22:10,22
2,18:45,18,20:55,20
3,19:40,19,21:50,21
4,20:00,20,22:10,22


In [16]:
df['start_hour'] = pd.cut(
    df['start_hour'],
    bins=[0, 3, 9, 15, 21, 24],
    labels=['EarlyMorning', 'Morning', 'Afternoon', 'Evening', 'LateNight'],
    include_lowest=True
)
df['end_hour'] = pd.cut(
    df['end_hour'],
    bins=[0, 3, 9, 15, 21, 24],
    labels=['EarlyMorning', 'Morning', 'Afternoon', 'Evening', 'LateNight'],
    include_lowest=True
)
df[['start_time', 'start_hour', 'end_time', 'end_hour']].head()

,start_time,start_hour,end_time,end_hour
0,21:05,Evening,23:15,LateNight
1,20:05,Evening,22:10,LateNight
2,18:45,Evening,20:55,Evening
3,19:40,Evening,21:50,Evening
4,20:00,Evening,22:10,LateNight


In [17]:
time_parts = df['trip_time'].str.extract(r'(?:(?P<hour>\d+)h)?\s*(?:(?P<minute>\d+)m)?')
time_parts = time_parts.astype(float).fillna(0)

df['trip_hour'] = time_parts['hour'] + time_parts['minute'] / 60

df[['trip_time', 'trip_hour']].value_counts()

trip_time  trip_hour
2h 10m     2.166667     11294
2h 5m      2.083333      7880
2h 0m      2.000000        95
2h 15m     2.250000        26
1h 20m     1.333333        17
1h 25m     1.416667        12
55m        0.916667         6
1h 55m     1.916667         6
1h 35m     1.583333         2
1h 0m      1.000000         1
1h 5m      1.083333         1
2h 25m     2.416667         1
Name: count, dtype: int64

In [18]:
df.drop(columns=['start_time', 'end_time', 'trip_time'], inplace=True)

## day

In [19]:
df['start_day'].unique(), df['start_day'].dtype, df['end_day'].unique(), df['end_day'].dtype

(array(['02 thg 5', '03 thg 5', '04 thg 5', '05 thg 5', '06 thg 5',
        '07 thg 5', '08 thg 5', '09 thg 5', '10 thg 5', '11 thg 5',
        '21 thg 4', '22 thg 4', '23 thg 4', '24 thg 4', '25 thg 4',
        '26 thg 4', '27 thg 4', '28 thg 4', '29 thg 4', '30 thg 4',
        '01 thg 5'], dtype=object),
 dtype('O'),
 array(['02 thg 5', '03 thg 5', '04 thg 5', '05 thg 5', '06 thg 5',
        '07 thg 5', '08 thg 5', '09 thg 5', '10 thg 5', '11 thg 5',
        '12 thg 5', '22 thg 4', '21 thg 4', '23 thg 4', '24 thg 4',
        '25 thg 4', '26 thg 4', '27 thg 4', '28 thg 4', '29 thg 4',
        '30 thg 4', '01 thg 5'], dtype=object),
 dtype('O'))

In [20]:
df[['start_day', 'end_day']].head()

,start_day,end_day
0,02 thg 5,02 thg 5
1,02 thg 5,02 thg 5
2,02 thg 5,02 thg 5
3,02 thg 5,02 thg 5
4,02 thg 5,02 thg 5


In [21]:
def convert_vn_date(date_str, year=2025):
    day, month = date_str.strip().split(' thg ')
    dt = pd.to_datetime(f"{day}-{int(month):02d}-{year}", dayfirst=True)
    return dt

df['start_day'] = df['start_day'].apply(lambda x: convert_vn_date(x, 2025))
df['end_day'] = df['end_day'].apply(lambda x: convert_vn_date(x, 2025))
df[['start_day', 'end_day']].head(), df['start_day'].dtype, df['end_day'].dtype

(   start_day    end_day
 0 2025-05-02 2025-05-02
 1 2025-05-02 2025-05-02
 2 2025-05-02 2025-05-02
 3 2025-05-02 2025-05-02
 4 2025-05-02 2025-05-02,
 dtype('<M8[ns]'),
 dtype('<M8[ns]'))

In [22]:
holidays = [
    pd.Timestamp('2025-04-30').date(),
    pd.Timestamp('2025-05-01').date(),
]

nearby_holidays = [
    pd.Timestamp('2025-04-29').date(),
    pd.Timestamp('2025-05-02').date(),
    pd.Timestamp('2025-05-03').date(),
    pd.Timestamp('2025-05-04').date(),
]

def is_holiday(date):
    d = date.date()
    if d in holidays:
        return 3
    elif d in nearby_holidays:
        return 2
    elif d.weekday() >= 5:  # Saturday = 5
        return 1
    else:
        return 0
    
df['is_holiday'] = df['start_day'].apply(is_holiday)
df[['start_day', 'is_holiday']].value_counts().head(5)

start_day   is_holiday
2025-04-27  1             1136
2025-04-29  2             1058
2025-04-28  0             1051
2025-05-02  2             1021
2025-04-30  3             1000
Name: count, dtype: int64

In [23]:
df['crawl_date'] = pd.to_datetime(df['crawl_date'], dayfirst=True).dt.date
df[['start_day', 'crawl_date']].head()

,start_day,crawl_date
0,2025-05-02,2025-05-01
1,2025-05-02,2025-05-01
2,2025-05-02,2025-05-01
3,2025-05-02,2025-05-01
4,2025-05-02,2025-05-01


In [24]:
df['days_left'] = (pd.to_datetime(df['start_day']) - pd.to_datetime(df['crawl_date'])).dt.days
df[['start_day', 'crawl_date', 'days_left']].value_counts().head()

start_day   crawl_date  days_left
2025-05-04  2025-04-22  12           61
2025-04-27  2025-04-16  11           61
            2025-04-19  8            60
            2025-04-23  4            60
2025-04-25  2025-04-09  16           59
Name: count, dtype: int64

In [25]:
df.drop(columns=['start_day', 'end_day', 'crawl_date'], inplace=True)

## Take_place, Destination

In [26]:
df['take_place'].unique(), df['take_place'].dtype

(array(['TP HCM (SGN)\nSân bay Tân Sơn Nhất',
        'TP HCM (SGN)\nSân bay Tân Sơn Nhất\nNhà ga 1',
        'TP HCM (SGN)\nSân bay Tân Sơn Nhất\nNhà ga T3',
        'TP HCM (SGN)\nSân bay Tân Sơn Nhất\nNhà ga 3'], dtype=object),
 dtype('O'))

In [27]:
df.drop('take_place', axis=1, inplace=True)

In [28]:
df['destination'].unique()

array(['Hà Nội (HAN)\nSân bay Nội Bài',
       'Hà Nội (HAN)\nSân bay Nội Bài\nNhà ga 1',
       'Huế (HUI)\nSân bay quốc tế Phú Bài',
       'Đà Nẵng (DAD)\nSân bay Đà Nẵng',
       'Đà Lạt (DLI)\nSân bay Liên Khương',
       'Đồng Hới (VDH)\nSân bay Đồng Hới',
       'Nha Trang (CXR)\nSân bay Cam Ranh'], dtype=object)

In [29]:
len(df)

19341

In [30]:
valid_destinations  = ['Hà Nội (HAN)\nSân bay Nội Bài', 'Hà Nội (HAN)\nSân bay Nội Bài\nNhà ga 1']
df = df[df['destination'].isin(valid_destinations)]
df['destination'].value_counts()

destination
Hà Nội (HAN)\nSân bay Nội Bài              10364
Hà Nội (HAN)\nSân bay Nội Bài\nNhà ga 1     8938
Name: count, dtype: int64

In [31]:
len(df)

19302

In [32]:
df = df.drop(['destination'], axis=1)

## luggage:

In [ ]:
df['hand_luggage_kg'] = df['hand_luggage'].str.extract(r'(\d+)\s*kg').astype(float)
df['checked_baggage_kg'] = df['checked_baggage'].str.extract(r'(\d+)\s*kg').astype(float)
df[['hand_luggage', 'hand_luggage_kg', 'checked_baggage', 'checked_baggage_kg']].value_counts()

hand_luggage                hand_luggage_kg  checked_baggage    checked_baggage_kg
Hành lý xách tay 7 kg       7.0              Hành lý 0 kg       0.0                   8044
Hành lý xách tay 1 x 12 kg  12.0             Hành lý 1 x 23 kg  23.0                  4939
                                             Hành lý 23 kg      23.0                   922
Hành lý xách tay 7 kg       7.0              Hành lý 20 kg      20.0                   345
Hành lý xách tay 10 kg      10.0             Hành lý 1 x 23 kg  23.0                   292
Name: count, dtype: int64

In [39]:
df['hand_luggage'] = df['hand_luggage_kg']
df['checked_baggage'] = df['checked_baggage_kg']
df.drop(columns=['hand_luggage_kg', 'checked_baggage_kg'], inplace=True)

In [34]:
# df['luggage'] = df['hand_luggage_kg'].fillna(0) + df['checked_baggage_kg'].fillna(0)
# df['luggage'].value_counts()

In [35]:
# df.drop(['checked_baggage', 'hand_luggage', 'checked_baggage_kg', 'hand_luggage_kg'], axis=1, inplace=True)

## organizing

In [40]:
df.columns

Index(['brand', 'price', 'hand_luggage', 'checked_baggage', 'id', 'start_hour',
       'end_hour', 'trip_hour', 'is_holiday', 'days_left'],
      dtype='object')

In [41]:
order = ['id', 'brand', 'price', 'start_hour', 'end_hour', 'trip_hour', 'hand_luggage', 'checked_baggage', 'is_holiday', 'days_left']
df = df[order]
df.head()

,id,brand,price,start_hour,end_hour,trip_hour,hand_luggage,checked_baggage,is_holiday,days_left
0,HAN0106,Bamboo Airways,2852590,Evening,LateNight,2.166667,7.0,0.0,2,1
1,HAN0624,VietJet Air,2770744,Evening,LateNight,2.083333,7.0,0.0,2,1
2,HAN0555,VietJet Air,2770744,Evening,Evening,2.166667,7.0,0.0,2,1
3,HAN0582,VietJet Air,2770744,Evening,Evening,2.166667,7.0,0.0,2,1
4,HAN0603,VietJet Air,2770744,Evening,LateNight,2.166667,7.0,0.0,2,1


In [42]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 19302 entries, 0 to 19358
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   id               19302 non-null  object  
 1   brand            19302 non-null  object  
 2   price            19302 non-null  int64   
 3   start_hour       19302 non-null  category
 4   end_hour         19302 non-null  category
 5   trip_hour        19302 non-null  float64 
 6   hand_luggage     14542 non-null  float64 
 7   checked_baggage  14542 non-null  float64 
 8   is_holiday       19302 non-null  int64   
 9   days_left        19302 non-null  int64   
dtypes: category(2), float64(3), int64(3), object(2)
memory usage: 1.4+ MB


In [43]:
df.to_csv(f'{city}_preprocessed.csv', index=False)